### Building a Flight Deal Finder with APIs, Google Sheets, and SMS Notifications

I began building a Flight Deal Finder application that automatically searches for affordable flights and notifies users whenever it discovers a great deal. This project combines multiple REST APIs to create a real-world travel application. I learned how to retrieve airport and destination data from Google Sheets using the Sheety API, search for flight prices through a flight search API, process the returned flight information to identify the cheapest options, and send SMS notifications using the Twilio API. Throughout this project, I also gained more experience working with JSON data, API authentication, HTTP requests, and object-oriented programming to build a practical automation tool.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

SERPAPI_API_KEY = os.getenv("SERPAPI_API_KEY")
SHEETY_ENDPOINT = os.getenv("SHETTY_ENDPOINT")

print("SerpAPI Key:", SERPAPI_API_KEY)
print("Sheety Endpoint:", SHEETY_ENDPOINT)

ModuleNotFoundError: No module named 'dotenv'

step 3 solution

In [2]:
import os
import requests
from requests.auth import HTTPBasicAuth
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

SHEETY_PRICES_ENDPOINT = "https://api.sheety.co/your_endpoint/prices"


class DataManager:

    def __init__(self):
        self._user = os.environ["SHEETY_USERNAME"]
        self._password = os.environ["SHEETY_PASSWORD"]
        self._authorization = HTTPBasicAuth(self._user, self._password)
        self.destination_data = {}

    def get_destination_data(self):
        # 2. Use the Sheety API to GET all the data in that sheet and print it out.
        response = requests.get(url=SHEETY_PRICES_ENDPOINT, auth=self._authorization)
        data = response.json()
        self.destination_data = data["prices"]
        # print(data)
        return self.destination_data

ModuleNotFoundError: No module named 'dotenv'

In [ ]:
class FlightData:
    #This class is responsible for structuring the flight data.
    pass

flight search

In [ ]:
import os
import requests
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

SERPAPI_ENDPOINT = "https://serpapi.com/search"


class FlightSearch:

    def __init__(self):
        self._api_key = os.environ["SERPAPI_API_KEY"]

    def check_flights(self, origin_city_code, destination_city_code, from_time, to_time):
        query = {
            "engine": "google_flights",
            "departure_id": origin_city_code,
            "arrival_id": destination_city_code,
            "outbound_date": from_time.strftime("%Y-%m-%d"),
            "return_date": to_time.strftime("%Y-%m-%d"),
            "type": "1",
            "adults": "1",
            "currency": "GBP",
            "api_key": self._api_key,
        }

        response = requests.get(url=SERPAPI_ENDPOINT, params=query)

        if response.status_code != 200:
            print(f"check_flights() response code: {response.status_code}")
            return None

        data = response.json()
        if "error" in data:
            print(f"API error: {data['error']}")
            return None
        return data

main

In [ ]:
import requests_cache
from pprint import pprint
from datetime import datetime, timedelta
from data_manager import DataManager
from flight_search import FlightSearch

# ==================== Conserve requests and preserve your free plan ====================
# Here we are not caching anything ending in *.sheety.co
# everything else is cached for 1 hour (3600 seconds). 
# feel free to experiment! 
requests_cache.install_cache(
    "flight_cache",
    urls_expire_after={
        "*.sheety.co*": requests_cache.DO_NOT_CACHE,
        "*": 3600,
    }
)

# ==================== Talk to Sheety ====================

data_manager = DataManager()
sheet_data = data_manager.get_destination_data()
pprint(sheet_data)

# ==================== Set the Dates ====================

tomorrow = datetime.now() + timedelta(days=1)
six_month_from_today = datetime.now() + timedelta(days=(6 * 30))

# ==================== Do a Flight Search ====================

flight_search = FlightSearch()

flights = flight_search.check_flights(
    origin_city_code="LHR",
    destination_city_code="CDG",
    from_time=tomorrow,
    to_time=six_month_from_today
)

pprint(flights)